# Notes 
- In this notebook, we load static data from disk (a Zarr archive). 
- The data is assumed to have a datamodel compliant with what the pipeline expects in terms of dimensions and coordinates.

- start time = '1990-03-13T00:00:00.000000000'
- end time = '2010-07-31T00:00:00.000000000'


In [ ]:
# Add automatic reloading of modules in case of changes
%load_ext autoreload
%autoreload 2
from datetime import datetime, timezone
from pathlib import Path

from app import create_app

from dpyverification.configuration import Config, GeneralInfoConfig
from dpyverification.configuration.utils import (
    ForecastPeriods,
    Range,
    TimeUnits,
    VerificationPair,
    VerificationPeriod,
)
from dpyverification.constants import DataType
from dpyverification.datasources import NetCDFConfig
from dpyverification.pipeline import run_pipeline
from dpyverification.scores import CrpsForEnsembleConfig, RankHistogramConfig

data_dir = Path("C:/Git/data-conversion-notebook/data/compliant/")

general = GeneralInfoConfig(
    verification_period=VerificationPeriod(
        start=datetime(1990, 3, 1, tzinfo=timezone.utc),
        end=datetime(2010, 7, 31, tzinfo=timezone.utc),
        dimension="forecast_reference_time",
    ),
    verification_pairs=[
        VerificationPair(id="raw_raw", obs="obs", sim="raw_raw"),
        VerificationPair(id="lin_log", obs="obs", sim="lin_log"),
        VerificationPair(id="qqt_qqt", obs="obs", sim="qqt_qqt"),
    ],
    forecast_periods=ForecastPeriods(unit=TimeUnits.day, values=Range(start=0, end=10, step=1)),
)
config = Config(
    fileversion="0.1.0",
    general=general,
    datasources=[
        NetCDFConfig(
            general=general,
            import_adapter="netcdf",
            source="obs",
            data_type=DataType.observed_historical,
            directory=str(data_dir),
            filename_glob="obs_Q.nc",
        ),
        NetCDFConfig(
            general=general,
            import_adapter="netcdf",
            source="lin_log",
            data_type=DataType.simulated_forecast_ensemble,
            directory=str(data_dir),
            filename_glob="case-lin-log_Q.nc",
        ),
        NetCDFConfig(
            general=general,
            import_adapter="netcdf",
            source="raw_raw",
            data_type=DataType.simulated_forecast_ensemble,
            directory=str(data_dir),
            filename_glob="case-raw-raw_Q.nc",
        ),
        NetCDFConfig(
            general=general,
            import_adapter="netcdf",
            source="qqt_qqt",
            data_type=DataType.simulated_forecast_ensemble,
            directory=str(data_dir),
            filename_glob="case-qqt-qqt_Q.nc",
        ),
    ],
    scores=[
        # Don't reduce any dimensions for the CRPS, as we want to be able to analyze it along
        # all dimensions (e.g. forecast_reference_time)
        CrpsForEnsembleConfig(score_adapter="crps_for_ensemble", general=general, reduce_dims=[]),
        # Reduce the forecast_reference_time dimension for the rank histogram, as we want to analyze
        # the overall rank histogram over the whole verification period
        RankHistogramConfig(
            score_adapter="rank_histogram",
            general=general,
            reduce_dims=["forecast_reference_time"],
        ),
    ],
)

In [ ]:
output_dataset = run_pipeline(config)

In [ ]:
output_dataset.verification_pairs

In [ ]:
ds = output_dataset.get(verification_pair=config.general.verification_pairs[2])
ds

In [ ]:
app = create_app(output_dataset)
app.run(jupyter_mode="inline", jupyter_height=1400)